# Vinograd, Nair et al. (2024): fivefold cross-validation of the rSLDS fits

Paper: Vinograd A, Nair A, et al. (2024), Nature 634, 910-918,
[doi:10.1038/s41586-024-07915-x](https://doi.org/10.1038/s41586-024-07915-x). Dataset:
[DANDI:001037](https://dandiarchive.org/dandiset/001037/0.260609.1402), version 0.260609.1402 (the
most recent published version and the only one with all five miniscope mice; see
`stored_weights_and_stimulation.ipynb` for the versions and the table of all claims).

Methods, Dynamical system models of neural data: "We used two-dimensional models, selecting the
optimal number of states through fivefold cross-validation"; "We cross-validated the model using
fivefold cross-validation, for which we trained the data on four arbitrary portions of the data
and tested on a left out fifth portion"; "Models fit to miniscope data during engagement of
aggression obtained a cvR2 = 84.7 +/- 0.03%, while the same model explains 67.2 +/- 0.02% of
variance in data obtained from observation of aggression."

This notebook tests methods-cvr2 and runs the cross-validated reading of the number of states for
ed1n-engaging-x1-slower, ed1p-observing-x1-slower and fig1e-2p-x1-slower (their main reading, the
sample code's number of states, is in `rslds_fits.ipynb`). Folds are five contiguous blocks; the
random-frame reading the claims file also names needs a fit with masked frames, which the ssm
library's initialization does not accept (it fails on the mask), so it is not run. Numbers of
states tried: 2, 3 and 4, with the sample code's other settings and seed 0. The held-out block's
latents are inferred with the trained parameters fixed; the score is the R2 of the activity
predicted from them (neural R2), and, as a second reading, the sample code's R2 of the latents
against their forward prediction (latent R2).

In [ ]:
import time
import warnings

import numpy as np

import linattr as L

warnings.filterwarnings("ignore")
started = time.time()
print(L.ensure_ssm())
MS = {s: L.load_miniscope(s) for s in L.MINISCOPE}
TP = {s: L.load_2p(s) for s in L.TWO_PHOTON}
SESS = {("mini", s, c): MS[s][c]["Y"] for s in L.MINISCOPE for c in ["engage", "observe"]}
SESS.update({("2p", s, "auto"): TP[s]["sessions"]["observation"] for s in L.TWO_PHOTON})
print(f"dandiset {L.DANDISET} version {L.VERSION}: {len(SESS)} sessions loaded in {time.time() - started:.0f} s")

## Data check

In [ ]:
for k, Y in SESS.items():
    blocks = np.array_split(np.arange(len(Y)), 5)
    print(f"{k[0]:4s} {k[1]:6s} {k[2]:7s}: {Y.shape[0]} frames x {Y.shape[1]} cells; mean {Y.mean():.3f}, sd {Y.std():.3f}; "
          f"fold lengths {[len(b) for b in blocks]} frames")

Every session is split into five contiguous blocks of 30 to 142 s. The same arrays as in the other
notebooks are used (z-scored, mean 0, SD near 1).

In [ ]:
KS = [2, 3, 4]
jobs = []
for k, Y in SESS.items():
    base = L.MINI_KW if k[0] == "mini" else L.TWO_P_KW
    for K in KS:
        for fold in range(5):
            jobs.append(((k, K, fold), Y, dict(base, K=K, seed=0), fold))
t0 = time.time()
CV = L.run_cv(jobs)
print(f"{len(CV)} cross-validation fits in {(time.time() - t0) / 60:.1f} min")


def cv_mean(k, K, measure):
    return float(np.mean([CV[(k, K, f)][measure] for f in range(5)]))


print("Held-out R2 by number of states (mean over 5 folds):")
for k in SESS:
    print(f"  {k[0]:4s} {k[1]:6s} {k[2]:7s} neural {[round(cv_mean(k, K, 'neural_r2'), 3) for K in KS]}; "
          f"against the session mean {[round(cv_mean(k, K, 'neural_r2_session'), 3) for K in KS]}; "
          f"latent {[round(cv_mean(k, K, 'latent_r2'), 3) for K in KS]}")

## methods-cvr2

> rSLDS models of miniscope data explain 84.7% of variance during engagement and 67.2% during
> observation of aggression (cross-validated R2).

The model is the sample code's miniscope model (3 states).

In [ ]:
for measure in ["neural_r2", "neural_r2_session", "latent_r2"]:
    for c, exp, tol in [("engage", 84.7, 3), ("observe", 67.2, 2)]:
        v = np.array([100 * cv_mean(("mini", s, c), 3, measure) for s in L.MINISCOPE])
        folds = np.array([[100 * CV[(("mini", s, c), 3, f)][measure] for f in range(5)] for s in L.MINISCOPE])
        print(f"{measure:9s} {c:7s}: per mouse {np.round(v, 1).tolist()} %; mean {v.mean():.1f} +/- {v.std(ddof=1) / np.sqrt(5):.1f} % "
              f"(expected {exp} +/- {tol}); fold range {folds.min():.1f} to {folds.max():.1f} %; mice at 70% or more: {(v >= 70).sum()} of 5")
best = {c: np.mean([100 * max(cv_mean(("mini", s, c), K, "neural_r2") for K in KS) for s in L.MINISCOPE]) for c in ["engage", "observe"]}
print("Variant, the best number of states per session (neural R2):", {c: round(v, 1) for c, v in best.items()})

**Result: blocked (methods underspecified).** No reading comes near the paper's values. With the
sample code's 3-state model and five contiguous folds, the held-out activity predicted from the
inferred latents explains -81.1% (engaging) and -74.5% (observing) of the variance about each held-out
block's own mean, and 12.5% and 19.8% about the session mean (expected 84.7 +/- 3 and 67.2 +/- 2%); the
sample code's latent R2 on the held-out blocks is negative in most mice. No mouse reaches the Methods'
70% threshold under any measure. A two-dimensional model cannot explain 85% of the variance of 42 to
158 neurons, so the paper's cvR2 must be taken on another quantity, which the paper does not name;
the random-frame folds could not be run (ssm does not accept masked frames at initialization).

## The cross-validated number of states, for ed1n, ed1p and fig1e

For each session the number of states with the highest mean held-out neural R2 is chosen, and the
model is refit to the whole session with it (seed 0). Time constants and the test are as in
`rslds_fits.ipynb`.

In [ ]:
choice = {k: KS[int(np.argmax([cv_mean(k, K, "neural_r2") for K in KS]))] for k in SESS}
print("Chosen number of states:", {f"{k[1]} {k[2]}": K for k, K in choice.items()})
jobs = [(k, SESS[k], dict(L.MINI_KW if k[0] == "mini" else L.TWO_P_KW, K=choice[k], seed=0), None) for k in SESS]
REFIT = L.run_fits(jobs)
for label, keys in [("ed1n engaging", [("mini", s, "engage") for s in L.MINISCOPE]),
                    ("ed1p observing", [("mini", s, "observe") for s in L.MINISCOPE]),
                    ("fig1e 2P observing", [("2p", s, "auto") for s in L.TWO_PHOTON])]:
    sf = [L.slow_fast(REFIT[k]) for k in keys]
    x1 = np.array([r["tau_x1"] for r in sf]); x2 = np.array([r["tau_x2"] for r in sf])
    las = np.array([r["las"] for r in sf])
    print(f"{label}: K {[choice[k] for k in keys]}; x1 {np.round(x1, 1).tolist()} s; x2 {np.round(x2, 1).tolist()} s; "
          f"x1 > x2 in {(x1 > x2).sum()} of {len(x1)}; Mann-Whitney p = {L.mwu(x1, x2):.4f}; scores {np.round(las, 2).tolist()}")
print(f"\nNotebook run time {(time.time() - started) / 60:.1f} min")

**Result: the cross-validated reading gives the same outcomes as the sample code's number of
states.** The held-out neural R2 hardly changes with the number of states (differences under 0.002),
so the chosen number is decided by small differences. With the chosen numbers, x1 is slower than x2 in
3 of 5 engaging sessions (Mann-Whitney p = 0.14), 3 of 5 observation sessions (p = 0.21) and 5 of 7
two-photon sessions (p = 0.31); ed1n-engaging-x1-slower, ed1p-observing-x1-slower and
fig1e-2p-x1-slower are not reproduced under this reading either.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| methods-cvr2 | blocked: methods underspecified (every reading run, results above) |
| ed1n-engaging-x1-slower, ed1p-observing-x1-slower, fig1e-2p-x1-slower | cross-validated reading not reproduced, as the main reading in `rslds_fits.ipynb` |

How the analysis differs from the paper's: contiguous folds only; 2 to 4 states; the measure of
variance explained is not given in the paper and three are shown.